# <center> Homework 125

In [4]:
import tensorflow_datasets as tfds
from importlib import reload
import tf_data
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_images
import numpy as np

## Task 0

да се имплементира stratify за Dataset

In [11]:
datasets = tfds.load(name="mnist")
mnist_train, mnist_test = datasets["train"], datasets["test"]

In [19]:
reload(tf_data)
from tf_data import Dataset

def counter(state, item):
    state[item[1].numpy()] += 1 
    return state

custom_ds = Dataset(mnist_train.take(10_000)).map(lambda x: (x['image'], x['label']))
classes_counts = custom_ds.reduce(np.zeros(10), counter)

total_samples = custom_ds.reduce(0, lambda state, _: state + 1)
class_ratio = classes_counts / total_samples

for cls, ratio in zip(range(10), class_ratio):
    print(cls, '->', ratio)

0 -> 0.1019
1 -> 0.1095
2 -> 0.0996
3 -> 0.1002
4 -> 0.0971
5 -> 0.0896
6 -> 0.0994
7 -> 0.1079
8 -> 0.0964
9 -> 0.0984


In [60]:
reload(tf_data)
from tf_data import Dataset
n_classes = 10
custom_ds = Dataset(mnist_train.take(10_000)).map(lambda x: (x['image'], x['label']))
train_ds, valid_ds = custom_ds.stratify(n_classes=n_classes, train_set_ratio=0.8)

train_classes_counts = train_ds.reduce(np.zeros(n_classes), counter)
valid_classes_counts = valid_ds.reduce(np.zeros(n_classes), counter)

train_samples = train_ds.reduce(0, lambda state, _: state + 1)
valid_samples = valid_ds.reduce(0, lambda state, _: state + 1)

train_classes_ratio = train_classes_counts / train_samples
valid_classes_ratio = valid_classes_counts / valid_samples

print(f'Train set: {train_samples} samples')
for cls, ratio in zip(range(n_classes), train_classes_ratio):
    print(cls, '->', ratio)


print(f'\nValid set: {valid_samples} samples')
for cls, ratio in zip(range(n_classes), valid_classes_ratio):
    print(cls, '->', ratio)

Train set: 7996 samples
0 -> 0.10192596298149074
1 -> 0.10955477738869435
2 -> 0.09954977488744372
3 -> 0.10017508754377188
4 -> 0.09704852426213106
5 -> 0.0895447723861931
6 -> 0.09942471235617809
7 -> 0.10792896448224112
8 -> 0.0964232116058029
9 -> 0.09842421210605302

Valid set: 2004 samples
0 -> 0.10179640718562874
1 -> 0.1092814371257485
2 -> 0.0998003992015968
3 -> 0.10029940119760479
4 -> 0.09730538922155689
5 -> 0.08982035928143713
6 -> 0.09930139720558882
7 -> 0.10778443113772455
8 -> 0.09630738522954092
9 -> 0.09830339321357286


## Task 1

да се имплементира RNN която предсказава следващата буква от аззбуката като се ползва вградения в Keras клас SimpleRNN

Пълна поредица

A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z

Ако дължината на frame-a е 6 символа, тестовите данни и target-ите ще изгледат така:

A, B, C, D, E, F => B, C, D, E, F, G

B, C, D, E, F, G => C, D, E, F, G, H

...

U, V, W, X, Y, Z => V, W, X, Y, Z, A


In [91]:
alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
char_to_inx = {char: i for i, char in enumerate(alphabet)}

frame_len = 6
X, y = [], []

for i in range(len(alphabet)):
    start, end = i, i + frame_len
    X.append([char_to_inx[char] for char in alphabet[start:end] + alphabet[:max((i + frame_len) - len(alphabet), 0)]])
    y.append(char_to_inx[alphabet[end % len(alphabet)]])

In [93]:
y

[6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 0,
 1,
 2,
 3,
 4,
 5]

In [94]:
X = tf.convert_to_tensor(X)
y = tf.convert_to_tensor(y)

In [95]:
X.shape

TensorShape([26, 6])

In [96]:
rnn = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(alphabet), output_dim=10, input_length=frame_len),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(len(alphabet), activation='softmax')
])

rnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [97]:
rnn.fit(X, y, batch_size=3, epochs=100)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.0000e+00 - loss: 3.2710
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0385 - loss: 3.2326    
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1154 - loss: 3.2030  
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3462 - loss: 3.1714     
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4615 - loss: 3.1312 
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4615 - loss: 3.0802
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5385 - loss: 3.0163
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5385 - loss: 2.9258
Epoch 9/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5385 - loss: 2.8069 
Epoch 10/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5385 - loss: 2.6573 
Epoch 11/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6154 - loss: 2.4812
Epoch 12/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accurac

In [84]:
inx_to_char = {i: char for i, char in enumerate(alphabet)}

In [98]:
seq = 'EFGHIJ'
X_new = tf.constant([[char_to_inx[char] for char in seq]])

pred = rnn.predict(X_new)[0]
inx = tf.argmax(pred)
inx_to_char[int(inx)] # correct K

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step


'K'

In [99]:
seq = 'WXYZAB'
X_new = tf.constant([[char_to_inx[char] for char in seq]])

pred = rnn.predict(X_new)[0]
inx = tf.argmax(pred)
inx_to_char[int(inx)] # correct C

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step


'C'

In [ ]:
seq = 'UVWXYZ'
X_new = tf.constant([[char_to_inx[char] for char in seq]])

pred = rnn.predict(X_new)[0]
inx = tf.argmax(pred)
inx_to_char[int(inx)] # correct A

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


'A'